In [0]:
# ==========================================================
# TSI - FASE 5: FORMULACIÓN DEL ÍNDICE
# PASO 56.1: RANGOS DE REFERENCIA POR FUENTE
# ==========================================================

from pyspark.sql import functions as F

TABLA_MODELO = "workspace.tsi.international_modeling"

df = spark.table(TABLA_MODELO)

# Para esta primera formulación utilizaremos las
# fuentes con orientación monotónica documentada.
#
# Madrid se analizará por separado porque el flujo
# vehicular no representa directamente la congestión.

FUENTES_INICIALES = [
    "istanbul_traffic_index",
    "sao_paulo_cet",
    "metr_la",
    "pems_bay"
]

df_base = (
    df
    .filter(F.col("fuente").isin(FUENTES_INICIALES))
    .filter(F.col("valor_principal").isNotNull())
    .filter(
        ~(
            (F.col("fuente") == "metr_la")
            &
            (F.col("calidad_derivada") == "sospechoso_cero_global")
        )
    )
)

print("=== REGISTROS DISPONIBLES PARA FORMULACIÓN ===")

(
    df_base
    .groupBy("fuente")
    .count()
    .orderBy("fuente")
    .show(truncate=False)
)

print("\n=== RANGOS ESTADÍSTICOS POR FUENTE ===")

df_rangos = (
    df_base
    .groupBy("fuente", "metrica_principal")
    .agg(
        F.count("*").alias("observaciones"),
        F.min("valor_principal").alias("minimo"),
        F.expr(
            "percentile_approx(valor_principal, 0.05, 10000)"
        ).alias("p05"),
        F.expr(
            "percentile_approx(valor_principal, 0.50, 10000)"
        ).alias("mediana"),
        F.expr(
            "percentile_approx(valor_principal, 0.95, 10000)"
        ).alias("p95"),
        F.max("valor_principal").alias("maximo")
    )
    .orderBy("fuente")
)

df_rangos.show(truncate=False)

print("\n=== VALIDACIONES ===")

assert df_rangos.count() == 4

rangos_invalidos = (
    df_rangos
    .filter(
        F.col("p05").isNull()
        | F.col("p95").isNull()
        | (F.col("p95") <= F.col("p05"))
    )
    .count()
)

assert rangos_invalidos == 0

print("Rangos de referencia calculados correctamente.")
print("Todavía no se ha calculado ni guardado el TSI.")


In [0]:
# ==========================================================
# TSI - FASE 5: FORMULACIÓN DEL ÍNDICE
# PASO 56.2: CÁLCULO EXPERIMENTAL DEL TSI
# ==========================================================

from pyspark.sql import functions as F

# 1. Incorporar los rangos calculados anteriormente
df_tsi_experimental = (
    df_base
    .join(
        df_rangos.select(
            "fuente",
            "metrica_principal",
            "p05",
            "p95"
        ),
        on=["fuente", "metrica_principal"],
        how="inner"
    )
)

# 2. Normalización robusta
# Se limita el resultado al intervalo [0, 1].

normalizado = (
    (F.col("valor_principal") - F.col("p05"))
    / (F.col("p95") - F.col("p05"))
)

normalizado = F.greatest(
    F.lit(0.0),
    F.least(F.lit(1.0), normalizado)
)

# 3. Orientación del indicador
# Velocidad: mayor valor -> mayor estabilidad relativa.
# Congestión: mayor valor -> menor estabilidad relativa.

df_tsi_experimental = (
    df_tsi_experimental
    .withColumn(
        "tsi",
        F.when(
            F.col("orientacion_metrica")
            == "menor_valor_posible_deterioro",
            normalizado * 100.0
        )
        .when(
            F.col("orientacion_metrica")
            == "mayor_valor_mayor_congestion",
            (1.0 - normalizado) * 100.0
        )
    )
    .withColumn(
        "version_tsi",
        F.lit("TSI_EXP_V1")
    )
)

# 4. Validaciones

print("=== VALIDACIÓN DEL TSI EXPERIMENTAL ===")

total = df_tsi_experimental.count()

assert total == 28_814_111, (
    f"Total inesperado: {total}"
)

invalidos = (
    df_tsi_experimental
    .filter(
        F.col("tsi").isNull()
        | F.isnan("tsi")
        | (F.col("tsi") < 0)
        | (F.col("tsi") > 100)
    )
    .count()
)

assert invalidos == 0, (
    f"Se encontraron {invalidos} valores inválidos."
)

print(f"Observaciones procesadas: {total:,}")
print(f"Valores TSI inválidos: {invalidos:,}")

# 5. Resumen por fuente

print("\n=== RESUMEN DEL TSI POR FUENTE ===")

(
    df_tsi_experimental
    .groupBy("fuente")
    .agg(
        F.count("*").alias("observaciones"),
        F.round(F.min("tsi"), 2).alias("tsi_min"),
        F.round(F.avg("tsi"), 2).alias("tsi_promedio"),
        F.round(F.max("tsi"), 2).alias("tsi_max"),
        F.round(F.stddev("tsi"), 2).alias("tsi_std")
    )
    .orderBy("fuente")
    .show(truncate=False)
)

print("\nTSI experimental calculado en memoria.")
print("No se han modificado las tablas Delta.")

In [0]:
# ==========================================================
# TSI - FASE 5: FORMULACIÓN DEL ÍNDICE
# PASO 56.3: PERSISTENCIA DEL TSI EXPERIMENTAL V1
# ==========================================================

from pyspark.sql import functions as F

TABLA_TSI = "workspace.tsi.tsi_experimental_v1"

TOTAL_ESPERADO = 28_814_111

CONTEOS_ESPERADOS = {
    "istanbul_traffic_index": 3_332,
    "metr_la": 6_649_668,
    "pems_bay": 16_937_700,
    "sao_paulo_cet": 5_223_411,
}

print("=== VALIDACIÓN PREVIA ===")

total = df_tsi_experimental.count()

assert total == TOTAL_ESPERADO

conteos = {
    fila["fuente"]: fila["count"]
    for fila in (
        df_tsi_experimental
        .groupBy("fuente")
        .count()
        .collect()
    )
}

assert conteos == CONTEOS_ESPERADOS

print("Integridad del TSI confirmada.")

print("\n=== PERSISTENCIA EN DELTA LAKE ===")

if spark.catalog.tableExists(TABLA_TSI):

    print("La tabla ya existe.")
    print("No se realizará ninguna escritura.")

else:

    print("Creando tabla Delta del TSI experimental...")

    (
        df_tsi_experimental
        .select(
            "timestamp",
            "ciudad",
            "fuente",
            "segmento_id",
            "metrica_principal",
            "valor_principal",
            "unidad_original",
            "orientacion_metrica",
            "calidad_derivada",
            "p05",
            "p95",
            "tsi",
            "version_tsi"
        )
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLA_TSI)
    )

    print("Tabla creada correctamente.")

print("\n=== VALIDACIÓN DE LA TABLA GUARDADA ===")

df_guardado = spark.table(TABLA_TSI)

total_guardado = df_guardado.count()

assert total_guardado == TOTAL_ESPERADO

conteos_guardados = {
    fila["fuente"]: fila["count"]
    for fila in df_guardado.groupBy("fuente").count().collect()
}

assert conteos_guardados == CONTEOS_ESPERADOS

invalidos = (
    df_guardado
    .filter(
        F.col("tsi").isNull()
        | F.isnan("tsi")
        | (F.col("tsi") < 0)
        | (F.col("tsi") > 100)
    )
    .count()
)

assert invalidos == 0

print(f"Observaciones guardadas: {total_guardado:,}")
print(f"Valores inválidos: {invalidos:,}")

print("\n=== INFORMACIÓN DELTA ===")

(
    spark.sql(f"DESCRIBE DETAIL {TABLA_TSI}")
    .select("format", "numFiles")
    .show(truncate=False)
)

print("\nPERSISTENCIA VALIDADA CORRECTAMENTE")
print(f"Tabla: {TABLA_TSI}")

In [0]:
# ==========================================================
# TSI - FASE 5: FORMULACIÓN DEL ÍNDICE
# PASO 57.1: AGREGACIÓN TEMPORAL DEL TSI
# ==========================================================

from pyspark.sql import functions as F

TABLA_TSI = "workspace.tsi.tsi_experimental_v1"

df_tsi = spark.table(TABLA_TSI)

print("=== AGREGACIÓN TEMPORAL POR FUENTE ===")

df_tsi_horario = (
    df_tsi
    .withColumn(
        "hora",
        F.date_trunc("hour", F.col("timestamp"))
    )
    .groupBy(
        "fuente",
        "hora"
    )
    .agg(
        F.avg("tsi").alias("tsi_promedio"),
        F.stddev("tsi").alias("tsi_desviacion"),
        F.min("tsi").alias("tsi_min"),
        F.max("tsi").alias("tsi_max"),
        F.count("*").alias("observaciones"),
        F.countDistinct("segmento_id").alias("segmentos")
    )
    .withColumn(
        "version_tsi",
        F.lit("TSI_EXP_V1")
    )
)

print("\n=== REGISTROS AGREGADOS POR FUENTE ===")

(
    df_tsi_horario
    .groupBy("fuente")
    .agg(
        F.count("*").alias("horas"),
        F.min("hora").alias("inicio"),
        F.max("hora").alias("fin")
    )
    .orderBy("fuente")
    .show(truncate=False)
)

print("\n=== VALIDACIÓN ===")

total_horas = df_tsi_horario.count()

invalidos = (
    df_tsi_horario
    .filter(
        F.col("tsi_promedio").isNull()
        | F.isnan("tsi_promedio")
        | (F.col("tsi_promedio") < 0)
        | (F.col("tsi_promedio") > 100)
    )
    .count()
)

assert total_horas > 0
assert invalidos == 0

print(f"Registros horarios: {total_horas:,}")
print(f"Promedios TSI inválidos: {invalidos:,}")

print("\n=== MUESTRA DE LA SERIE HORARIA ===")

(
    df_tsi_horario
    .orderBy("fuente", "hora")
    .show(12, truncate=False)
)

print("\nAgregación temporal completada en memoria.")
print("No se modificaron las tablas Delta.")

In [0]:
# ==========================================================
# TSI - FASE 5: FORMULACIÓN DEL ÍNDICE
# PASO 57.2: VALIDACIÓN DE CONTINUIDAD TEMPORAL
# ==========================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("=== CONTINUIDAD TEMPORAL POR FUENTE ===")

ventana = (
    Window
    .partitionBy("fuente")
    .orderBy("hora")
)

df_continuidad = (
    df_tsi_horario
    .withColumn(
        "hora_anterior",
        F.lag("hora").over(ventana)
    )
    .withColumn(
        "diferencia_horas",
        (
            F.unix_timestamp("hora")
            - F.unix_timestamp("hora_anterior")
        ) / 3600.0
    )
)

df_resumen_continuidad = (
    df_continuidad
    .groupBy("fuente")
    .agg(
        F.count("*").alias("registros"),
        F.sum(
            F.when(
                F.col("diferencia_horas") == 1,
                1
            ).otherwise(0)
        ).alias("intervalos_consecutivos"),
        F.sum(
            F.when(
                F.col("diferencia_horas") > 1,
                1
            ).otherwise(0)
        ).alias("saltos_temporales"),
        F.max("diferencia_horas").alias(
            "salto_maximo_horas"
        )
    )
    .withColumn(
        "continuidad_pct",
        F.round(
            F.col("intervalos_consecutivos")
            / (F.col("registros") - 1) * 100,
            2
        )
    )
)

df_resumen_continuidad.orderBy("fuente").show(
    truncate=False
)

print("\n=== VALIDACIÓN DE DUPLICADOS ===")

duplicados = (
    df_tsi_horario
    .groupBy("fuente", "hora")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicados == 0

print(f"Claves fuente-hora duplicadas: {duplicados}")

print("\n=== INTERPRETACIÓN ===")
print("100 % = todas las observaciones consecutivas están separadas por una hora.")
print("Valores menores indican saltos en la serie temporal.")
print("No se modificaron ni imputaron datos.")

In [0]:
# ==========================================================
# TSI - FASE 5: FORMULACIÓN DEL ÍNDICE
# PASO 57.3: PREPARACIÓN TEMPORAL DE PEMS-BAY
# ==========================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

FUENTE_OBJETIVO = "pems_bay"

# 1. Seleccionar la serie horaria
df_pems = (
    df_tsi_horario
    .filter(F.col("fuente") == FUENTE_OBJETIVO)
    .select(
        "fuente",
        "hora",
        "tsi_promedio",
        "tsi_desviacion",
        "observaciones",
        "segmentos"
    )
)

# 2. Definir orden cronológico
ventana = Window.orderBy("hora")

# 3. Obtener la observación de la siguiente hora
df_pems_supervisado = (
    df_pems
    .withColumn(
        "hora_siguiente",
        F.lead("hora").over(ventana)
    )
    .withColumn(
        "tsi_siguiente",
        F.lead("tsi_promedio").over(ventana)
    )
    .withColumn(
        "diferencia_horas",
        (
            F.unix_timestamp("hora_siguiente")
            - F.unix_timestamp("hora")
        ) / 3600.0
    )
)

# 4. Conservar solamente pares consecutivos
df_pems_supervisado = (
    df_pems_supervisado
    .filter(F.col("diferencia_horas") == 1)
    .select(
        "fuente",
        "hora",
        "tsi_promedio",
        "tsi_desviacion",
        "observaciones",
        "segmentos",
        "hora_siguiente",
        "tsi_siguiente"
    )
)

print("=== DATASET SUPERVISADO PEMS-BAY ===")

total = df_pems_supervisado.count()

print(f"Pares horarios consecutivos: {total:,}")

assert total == 4_341

# 5. Validar objetivos
invalidos = (
    df_pems_supervisado
    .filter(
        F.col("tsi_siguiente").isNull()
        | F.isnan("tsi_siguiente")
        | (F.col("tsi_siguiente") < 0)
        | (F.col("tsi_siguiente") > 100)
    )
    .count()
)

assert invalidos == 0

print(f"Objetivos inválidos: {invalidos:,}")

print("\n=== RANGO TEMPORAL ===")

(
    df_pems_supervisado
    .agg(
        F.min("hora").alias("inicio"),
        F.max("hora_siguiente").alias("fin")
    )
    .show(truncate=False)
)

print("\n=== MUESTRA ===")

(
    df_pems_supervisado
    .orderBy("hora")
    .show(10, truncate=False)
)

print("\nPreparación supervisada completada en memoria.")
print("No se modificaron tablas Delta.")

In [0]:
# ==========================================================
# TSI - FASE 5: FORMULACIÓN DEL ÍNDICE
# PASO 57.4: DIVISIÓN CRONOLÓGICA TRAIN / TEST
# ==========================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("=== PREPARACIÓN DE PARTICIONES TEMPORALES ===")

# 1. Ordenar cronológicamente los registros
ventana = Window.orderBy("hora")

df_ordenado = (
    df_pems_supervisado
    .withColumn(
        "indice_temporal",
        F.row_number().over(ventana)
    )
)

# 2. Calcular tamaño de las particiones
total = df_ordenado.count()

n_train = int(total * 0.80)
n_test = total - n_train

# 3. Separación temporal
df_train = (
    df_ordenado
    .filter(F.col("indice_temporal") <= n_train)
)

df_test = (
    df_ordenado
    .filter(F.col("indice_temporal") > n_train)
)

print("\n=== DISTRIBUCIÓN ===")
print(f"Total: {total:,}")
print(f"Entrenamiento: {df_train.count():,}")
print(f"Prueba: {df_test.count():,}")

# 4. Validar fechas
print("\n=== RANGOS TEMPORALES ===")

print("ENTRENAMIENTO")
df_train.agg(
    F.min("hora").alias("inicio"),
    F.max("hora").alias("fin")
).show(truncate=False)

print("PRUEBA")
df_test.agg(
    F.min("hora").alias("inicio"),
    F.max("hora").alias("fin")
).show(truncate=False)

# 5. Validaciones
assert total == 4_341
assert df_train.count() == n_train
assert df_test.count() == n_test

ultima_hora_train = (
    df_train.agg(F.max("hora")).first()[0]
)

primera_hora_test = (
    df_test.agg(F.min("hora")).first()[0]
)

assert ultima_hora_train < primera_hora_test

print("\nVALIDACIÓN CORRECTA")
print("División cronológica realizada sin mezclar fechas.")
print("No se modificaron tablas Delta.")

In [0]:
# ==========================================================
# TSI - FASE 5: FORMULACIÓN DEL ÍNDICE
# PASO 57.5: RANGOS AJUSTADOS SOLO CON TRAIN
# ==========================================================

from pyspark.sql import functions as F

TABLA_MODELO = "workspace.tsi.international_modeling"

# 1. Recuperar las fechas del entrenamiento
fecha_inicio_train = (
    df_train
    .agg(F.min("hora"))
    .first()[0]
)

fecha_fin_train = (
    df_train
    .agg(F.max("hora"))
    .first()[0]
)

fecha_inicio_test = (
    df_test
    .agg(F.min("hora"))
    .first()[0]
)

print("=== PARTICIÓN TEMPORAL ===")
print(f"Inicio train: {fecha_inicio_train}")
print(f"Última hora train: {fecha_fin_train}")
print(f"Primera hora test: {fecha_inicio_test}")

# 2. Recuperar las velocidades originales
df_pems_original = (
    spark.table(TABLA_MODELO)
    .filter(F.col("fuente") == "pems_bay")
    .select(
        "timestamp",
        "segmento_id",
        "valor_principal"
    )
)

# 3. Ajustar percentiles con datos de entrenamiento
#
# Se incluye la hora completa de la última observación
# de entrenamiento, pero no se utiliza ningún registro
# perteneciente al periodo de prueba.

df_referencia_train = (
    df_pems_original
    .filter(
        F.col("timestamp")
        < F.lit(fecha_inicio_test)
    )
)

print("\n=== RANGOS DE ENTRENAMIENTO ===")

rangos_train = (
    df_referencia_train
    .agg(
        F.count("*").alias("observaciones_train"),
        F.expr(
            "percentile_approx(valor_principal, 0.05, 10000)"
        ).alias("p05_train"),
        F.expr(
            "percentile_approx(valor_principal, 0.95, 10000)"
        ).alias("p95_train")
    )
)

rangos_train.show(truncate=False)

resultado = rangos_train.first()

p05_train = resultado["p05_train"]
p95_train = resultado["p95_train"]

assert p05_train is not None
assert p95_train is not None
assert p95_train > p05_train

print("\n=== VALIDACIÓN ===")
print(f"P05 train: {p05_train}")
print(f"P95 train: {p95_train}")
print("Los percentiles se ajustaron sin usar observaciones del periodo de prueba.")
print("No se modificaron tablas Delta.")

In [0]:
# ==========================================================
# TSI - FASE 5: FORMULACIÓN DEL ÍNDICE
# PASO 57.6: TSI NORMALIZADO CON TRAIN
# ==========================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Aplicar la referencia de entrenamiento
df_pems_tsi_train = (
    df_pems_original
    .withColumn(
        "tsi",
        F.least(
            F.lit(100.0),
            F.greatest(
                F.lit(0.0),
                F.lit(100.0)
                * (F.col("valor_principal") - F.lit(p05_train))
                / F.lit(p95_train - p05_train)
            )
        )
    )
)

# 2. Agregar por hora
df_pems_horario_train = (
    df_pems_tsi_train
    .withColumn(
        "hora",
        F.date_trunc("hour", F.col("timestamp"))
    )
    .groupBy("hora")
    .agg(
        F.avg("tsi").alias("tsi_promedio"),
        F.stddev("tsi").alias("tsi_desviacion"),
        F.count("*").alias("observaciones"),
        F.countDistinct("segmento_id").alias("segmentos")
    )
)

# 3. Construir pares temporales
ventana = Window.orderBy("hora")

df_pems_pares = (
    df_pems_horario_train
    .withColumn(
        "hora_siguiente",
        F.lead("hora").over(ventana)
    )
    .withColumn(
        "tsi_siguiente",
        F.lead("tsi_promedio").over(ventana)
    )
    .filter(
        F.col("hora_siguiente")
        == F.expr("hora + INTERVAL 1 HOUR")
    )
)

# 4. Separación cronológica sin cruzar la frontera
df_train_final = (
    df_pems_pares
    .filter(F.col("hora_siguiente") < F.lit(fecha_inicio_test))
)

df_test_final = (
    df_pems_pares
    .filter(F.col("hora") >= F.lit(fecha_inicio_test))
)

# 5. Validaciones
n_train_final = df_train_final.count()
n_test_final = df_test_final.count()

invalidos = (
    df_pems_pares
    .filter(
        F.col("tsi_promedio").isNull()
        | F.col("tsi_siguiente").isNull()
        | F.isnan("tsi_promedio")
        | F.isnan("tsi_siguiente")
        | (F.col("tsi_promedio") < 0)
        | (F.col("tsi_promedio") > 100)
        | (F.col("tsi_siguiente") < 0)
        | (F.col("tsi_siguiente") > 100)
    )
    .count()
)

assert invalidos == 0

assert (
    df_train_final
    .filter(F.col("hora_siguiente") >= F.lit(fecha_inicio_test))
    .count()
    == 0
)

assert (
    df_test_final
    .filter(F.col("hora") < F.lit(fecha_inicio_test))
    .count()
    == 0
)

print("=== DATASET SIN FUGA TEMPORAL ===")
print(f"Entrenamiento: {n_train_final:,}")
print(f"Prueba: {n_test_final:,}")
print(f"Valores inválidos: {invalidos:,}")

print("\n=== RANGOS TEMPORALES ===")

print("TRAIN")
df_train_final.agg(
    F.min("hora").alias("inicio"),
    F.max("hora_siguiente").alias("ultimo_objetivo")
).show(truncate=False)

print("TEST")
df_test_final.agg(
    F.min("hora").alias("inicio"),
    F.max("hora_siguiente").alias("ultimo_objetivo")
).show(truncate=False)

print("\n=== MUESTRA DEL NUEVO TSI ===")

(
    df_pems_pares
    .orderBy("hora")
    .select(
        "hora",
        "tsi_promedio",
        "hora_siguiente",
        "tsi_siguiente"
    )
    .show(10, truncate=False)
)

print("\nTSI reconstruido con percentiles de entrenamiento.")
print("No se modificaron tablas Delta.")

In [0]:
# ==========================================================
# TSI - FASE 6: EXPERIMENTACIÓN CON MLFLOW
# PASO 58.1: BASELINE DE PERSISTENCIA
# ==========================================================

import mlflow
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 1. Preparar los valores reales y predicciones
df_baseline = (
    df_test_final
    .select(
        "hora",
        "tsi_promedio",
        "tsi_siguiente"
    )
    .orderBy("hora")
    .toPandas()
)

y_real = df_baseline["tsi_siguiente"].to_numpy()
y_pred = df_baseline["tsi_promedio"].to_numpy()

# 2. Calcular métricas
mae = mean_absolute_error(y_real, y_pred)
rmse = np.sqrt(mean_squared_error(y_real, y_pred))
r2 = r2_score(y_real, y_pred)

print("=== BASELINE DE PERSISTENCIA ===")
print(f"Observaciones evaluadas: {len(y_real):,}")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")

assert len(y_real) == 869
assert np.isfinite([mae, rmse, r2]).all()

# 3. Configurar MLflow
EXPERIMENTO = "/Shared/TSI_PEMS_BAY_Pronostico_Horario"

mlflow.set_experiment(EXPERIMENTO)

# 4. Registrar experimento
with mlflow.start_run(
    run_name="baseline_persistencia_v1"
) as run:

    mlflow.log_params({
        "proyecto": "Traffic Stability Index",
        "fuente": "pems_bay",
        "modelo": "persistencia",
        "horizonte_horas": 1,
        "normalizacion": "percentiles_train",
        "p05_train": float(p05_train),
        "p95_train": float(p95_train),
        "train_registros": 3471,
        "test_registros": len(y_real),
        "version_tsi": "TSI_TRAIN_NORM_V1"
    })

    mlflow.log_metrics({
        "mae": float(mae),
        "rmse": float(rmse),
        "r2": float(r2)
    })

    run_id = run.info.run_id

print("\n=== REGISTRO MLFLOW ===")
print(f"Experimento: {EXPERIMENTO}")
print(f"Run ID: {run_id}")
print("Baseline registrado correctamente.")

In [0]:
# ==========================================================
# TSI - PASO 58.2A
# DIAGNÓSTICO DE SESIÓN Y TABLAS DELTA
# ==========================================================

variables = [
    "df_pems_original",
    "df_train_final",
    "df_test_final",
    "p05_train",
    "p95_train",
    "fecha_inicio_test"
]

print("=== VARIABLES EN MEMORIA ===")

for nombre in variables:
    disponible = nombre in globals()
    print(f"{nombre}: {'DISPONIBLE' if disponible else 'NO DISPONIBLE'}")

print("\n=== TABLAS DELTA ===")

tablas = [
    "workspace.tsi.international_modeling",
    "workspace.tsi.tsi_experimental_v1",
    "workspace.tsi.pems_bay_train_v1",
    "workspace.tsi.pems_bay_test_v1"
]

for tabla in tablas:
    existe = spark.catalog.tableExists(tabla)
    print(f"{tabla}: {'EXISTE' if existe else 'NO EXISTE'}")

print("\n=== REFERENCIAS DE NORMALIZACIÓN ===")

if "p05_train" in globals():
    print(f"P05 train: {p05_train}")

if "p95_train" in globals():
    print(f"P95 train: {p95_train}")

print("\nDiagnóstico completado. No se modificaron datos.")

In [0]:
# ==========================================================
# TSI - FASE 6: EXPERIMENTACIÓN CON MLFLOW
# PASO 58.2B: PERSISTENCIA CORREGIDA TRAIN / TEST
# ==========================================================

from pyspark.sql import functions as F

TABLA_TRAIN = "workspace.tsi.pems_bay_train_v1"
TABLA_TEST = "workspace.tsi.pems_bay_test_v1"

print("=== VALIDACIÓN PREVIA ===")

assert p05_train == 40.3
assert p95_train == 69.8

assert df_train_final.count() == 3471
assert df_test_final.count() == 869

print("Variables y conteos correctos.")

# 1. Guardar únicamente si la tabla no existe
datasets = [
    (TABLA_TRAIN, df_train_final, 3471),
    (TABLA_TEST, df_test_final, 869)
]

for nombre, df, esperado in datasets:

    print(f"\n=== PROCESANDO {nombre} ===")

    if spark.catalog.tableExists(nombre):
        print("La tabla ya existe. Se omite la escritura.")

    else:
        (
            df.write
            .format("delta")
            .mode("append")
            .saveAsTable(nombre)
        )

        print("Tabla Delta creada correctamente.")

    # Verificar registros persistidos
    total = spark.table(nombre).count()

    assert total == esperado, (
        f"Conteo incorrecto en {nombre}: {total}"
    )

    print(f"Registros validados: {total:,}")

# 2. Validar separación temporal
print("\n=== VALIDACIÓN TEMPORAL ===")

train = spark.table(TABLA_TRAIN)
test = spark.table(TABLA_TEST)

ultimo_objetivo_train = (
    train.agg(F.max("hora_siguiente")).first()[0]
)

primera_hora_test = (
    test.agg(F.min("hora")).first()[0]
)

assert ultimo_objetivo_train < primera_hora_test

print(f"Último objetivo train: {ultimo_objetivo_train}")
print(f"Primera hora test: {primera_hora_test}")

# 3. Comprobar formato Delta
print("\n=== INFORMACIÓN DELTA ===")

for nombre in [TABLA_TRAIN, TABLA_TEST]:
    print(f"\nTabla: {nombre}")

    spark.sql(
        f"DESCRIBE DETAIL {nombre}"
    ).select(
        "format",
        "numFiles"
    ).show(truncate=False)

print("\nPERSISTENCIA VALIDADA CORRECTAMENTE")
print("Datasets disponibles para experimentos con MLflow.")